# Airline AI Assistant

In [1]:
import os
from dotenv import load_dotenv
import gradio as gr
import json
from openai import OpenAI

/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")


In [3]:
if open_ai:
    print("OpenAI client initialized successfully.")
else:
    print("Failed to initialize OpenAI client. Please check your API key and environment variables.")

Model= "gpt-4.1-mini"
open_ai = OpenAI()

OpenAI client initialized successfully.


In [17]:
system_message = """You are a helpful assistant for a flight called FlightAI, 
Give short, courteous answers not more than 1 sentence to customer questions about their flight. 
If you don't know the answer, say you don't know"""

In [15]:
def chat(message, history):
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = open_ai.chat.completions.create(model=Model, messages=messages)
    return response.choices[0].message.content

In [18]:
gr.ChatInterface(fn=chat,title="FlightAI Assistant").launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


In [33]:
ticket_prices = {"london": 250, "paris": 300, "tokyo": 500, "new York": 400, "dubai": 350, "sydney": 450}
def get_ticket_prices(destination_city):
    print(f"Tool called for destination city: {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Sorry, we don't have information for that destination")
    return f"Price for the {destination_city} is {price}"

In [35]:

get_ticket_prices("LONDON")

Tool called for destination city: LONDON


'Price for the LONDON is 250'

In [51]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_prices",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [52]:
# And this is included in a list of tools:
tools = [{"type": "function", "function": price_function}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [53]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = open_ai.chat.completions.create(model=Model, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = open_ai.chat.completions.create(model=Model, messages=messages)
    
    return response.choices[0].message.content

In [54]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_prices":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_prices(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [55]:
gr.ChatInterface(fn=chat, title="Airline AI Assistant").launch()


* Running on local URL:  http://127.0.0.1:7883
* To create a public link, set `share=True` in `launch()`.


Tool called for destination city: London


Traceback (most recent call last):
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/pyth

## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [57]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_prices":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_prices(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [67]:
gr.ChatInterface(fn=chat, title="Airline AI Assistant").launch()

* Running on local URL:  http://127.0.0.1:7888
* To create a public link, set `share=True` in `launch()`.


Tool called for destination city: London
Tool called for destination city: Paris


In [66]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = open_ai.chat.completions.create(model=Model, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = open_ai.chat.completions.create(model=Model, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [68]:
import sqlite3

In [72]:
DB = "air_prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS airline_prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()

In [77]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM airline_prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [78]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [79]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO airline_prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [80]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [81]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [82]:
gr.ChatInterface(fn=chat, title="Airline AI Assistant").launch()

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.
